# 07 · The full robot agent loop & the path to a real robot

We now connect all the pieces: **voice command → VLM planner → guardrail → skills**. Then we map each simulated piece onto real hardware and the NVIDIA robotics stack.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
from edge_vlm.train import load_or_train, VLMPolicy, get_device
from edge_vlm.agent import FetchAgent
from edge_vlm.sim import FetchWorld

device = get_device()
model, tok = load_or_train(ROOT / 'checkpoints' / 'tiny_robot_vlm.pt', device=device)
policy = VLMPolicy(model, tok, device=device)

## 1. Watch an episode through the robot's eyes

In [ ]:
world = FetchWorld(seed=31)
command = world.command      # on the real robot this string comes from speech recognition
frames = []

class RecordingPolicy:
    def act(self, state):
        out = policy.act(state)
        frames.append((state.image, f'{state.room}\nholding: {state.holding}\n-> {out}'))
        return out

res = FetchAgent(RecordingPolicy()).run(world, command)
print('command:', command, '| success:', res.success)
fig, axes = plt.subplots(1, len(frames), figsize=(3 * len(frames), 3.6), squeeze=False)
for ax, (img, title) in zip(axes[0], frames):
    ax.imshow(img); ax.set_title(title, fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

## 2. The guardrail is not optional

Models hallucinate. Every output goes through `parse_action` (is it a known skill with valid arguments?) and `check_preconditions` (does it make sense *now*?). Only then do we execute it. On hardware, add more checks: a battery threshold, a human detected too close, gripper force limits, and a timeout per skill.

In [ ]:
from edge_vlm.actions import parse_action, check_preconditions
for raw in ['pick red cup', '{"skill": "navigate", "room": "garage"}', 'handover', 'sure! I will go to the kitchen', 'navigate kitchen']:
    a = parse_action(raw)
    verdict = 'REJECT (unparseable)' if a is None else check_preconditions(a, room='living_room', holding=None)
    print(f'{raw!r:45s} -> {a}  {verdict}')

## 3. Talk to it (text stands in for speech)

Try your own commands. The tiny model only knows its ~60-word vocabulary, so unknown words become `<unk>`. The SmolVLM policy from notebook 04 handles free-form phrasing.

In [ ]:
world = FetchWorld(seed=5)
world.reset(target='blue bottle', target_room='bedroom')
for cmd in ['bring me the blue bottle', 'go to the bedroom and get the blue bottle', 'fetch my blue bottle quickly']:
    world.reset(target='blue bottle', target_room='bedroom')
    r = FetchAgent(policy).run(world, cmd)
    print(f'{cmd!r:50s} success={r.success} steps={r.steps}  unknown words={[w for w in cmd.split() if w not in tok.stoi]}')

## 4. From simulation to the real robot

| Simulated piece | Real robot equivalent | NVIDIA / ROS 2 tools |
|---|---|---|
| `world.command` | Microphone + speech-to-text | **Riva** ASR, or Whisper (TensorRT / whisper.cpp) |
| `world.render()` | RGB(-D) camera | Isaac ROS camera drivers (Argus for CSI, RealSense/ZED/Orbbec ROS 2 drivers) |
| `obs.room` | Localization on a map with room labels | **Isaac ROS Visual SLAM (cuVSLAM)**, Nav2 AMCL; label room polygons on the map |
| `navigate <room>` / `return` | Drive to a room waypoint, avoid obstacles | **Nav2** + **Isaac ROS nvblox** (GPU 3D costmap) |
| `pick <obj>` | Detect → 6-DoF pose → grasp | Open-vocab detector (e.g. Grounding DINO / NanoOWL), **FoundationPose**, **cuMotion** / MoveIt 2 |
| `handover` | Arm presents the object, waits for a pull | arm controller + force/current sensing |
| `FetchAgent` loop | ROS 2 node | `deploy/ros2/vlm_planner_node.py` (skeleton) |

```
      /voice_command ─┐
/camera/image_raw ────┼─► vlm_planner (this repo, 1 Hz) ──► /robot/skill (JSON)
      /robot/state ───┘                                            │
                                        skill_executor ◄───────────┘
                          ┌───────────────┼───────────────┐
                     Nav2 goal       grasp pipeline     handover
                          └──────► /robot/state (room, holding, busy) ──► back to planner
```

### A realistic first hardware build
* **Compute:** Jetson Orin Nano Super dev kit (8 GB) with an NVMe SSD. Upgrade to Orin NX 16 GB if memory gets tight.
* **Base:** a differential-drive base with a ROS 2 driver and wheel odometry (many hobby/education kits exist).
* **Camera:** one RGB-D camera (depth is needed for grasping and the nvblox costmap).
* **Arm:** a small low-cost arm (for example the open-source SO-101 used by Hugging Face LeRobot) with a simple gripper.
* **Power:** a battery able to supply ~25 W for the Jetson plus the motors.

### Suggested milestones
1. Sim works end-to-end (this repo). ✅
2. Jetson runs the tiny model with TensorRT and you have measured latency and power (notebook 06).
3. Base drives to named rooms with Nav2 (no VLM yet, just publish skills by hand).
4. VLM planner on real camera images. Collect ~200 real teleop decision points and fine-tune SmolVLM (notebook 04 recipe).
5. Add the arm. Start with a fixed top-down grasp, then FoundationPose.
6. Iterate: real-world failure cases become new data, then a new fine-tune, then a new evaluation.
7. Stretch: replace skill selection + grasping with an end-to-end VLA (e.g. Isaac GR00T, LeRobot policies) and train it in **Isaac Lab**.